# Step 3: Language Detection

This notebook detects the languages used in comments on YouTube videos based on the results of step 2. for YouTube videos from work folders specified in "config/data_config.yml", creating (in the work folders containing the original files) 
1. comment tables with language identification data for each comment ("... _comments_cleaned_langdetect.csv")
2. metadata enriched with language diversity data for each dataset specified (_metadata_cleaned_langinfo.csv)

In [ ]:
import pandas as pd
import numpy as np
from glob import glob
import sys
import yaml
import os
import pprint
from collections import Counter
print(sys.executable)
print("NumPy version:", np.__version__)
# fasttext requires numpy < 2.0
print("Python location:", sys.executable)
import fasttext
model = fasttext.load_model("lid.176.bin")
print(model.predict("Hello, world!"))
from lingua import Language, LanguageDetectorBuilder
languages = [Language.ENGLISH, Language.FRENCH, Language.GERMAN, Language.SPANISH, Language.JAPANESE, Language.CHINESE, Language.KOREAN]
detector_lingua = LanguageDetectorBuilder.from_languages(*languages).build()
language = detector_lingua.detect_language_of("languages are awesome")
language

In [ ]:
# flags

dataset_config = "./config/dataset_config.yml"
catfile = '*combined_cleaned_data.csv'
output = './output/'
print(os.listdir(output))

In [ ]:
# data screening

print(os.path.abspath(dataset_config))
print(os.path.getsize(dataset_config))
with open(dataset_config, "r") as f:
    config = yaml.safe_load(f)
# Expand paths relative to working dir
directories = {
    key: {
        'wd': (
            [os.path.join(os.getcwd(), path) for path in value['wd']]
            if value['wd'] != None
            else []
        ),
        'catdir': (
            os.path.join(os.getcwd(), value['catdir'])
            if value['catdir'] != None
            else ''
        ),
    }
    for key, value in config.items()
}
for game in directories:
    tmpcatf =  glob(os.path.join(directories[game]['catdir'], catfile))
    if tmpcatf:
        directories[game]['catfile'] = tmpcatf[0]
        print(f'catfile detected: {directories[game]['catfile']}')
        directories[game]['catdata'] = pd.read_csv(directories[game]['catfile'])
    else:
        print('no category file exists')
        directories[game]['catfile'] = False

pprint.pprint(directories)

## Notes
load table to work on, the column with the comments needs to be named 'text'<br>
or change df['text'] to whatever the comments column is called in the table in the remove urls step below<br>
(NOTE: df['text2'] does not need to be changed)

In [ ]:
def identify_datafiles_in_directory(directory, tp):
    file_list = []
    print('searching for files in:', directory)
    for file_name in os.listdir(directory):
        if file_name.startswith('ytma') and file_name.endswith(tp):
            file_list.append(file_name)
    return file_list

gcld3, not used

In [ ]:
def detect_language_gcld3(df, text_column):
    def detect_lang(text):
        try:
            return pycld3.get_language(text).language
        except:
            print('EXCEPTION:', text)
            return None  # or '' if you prefer
    return df[text_column].astype(str).apply(detect_lang)

#df['gcld3'] = detect_language_gcld3(df, 'text_nourls')

lingua

In [ ]:
def detect_language_lingua(df, text_column, detector_lingua):
    def detect_lang(text):
        try:
            lang = detector_lingua.detect_language_of(text)
            if lang is None:
                return None
            return lang.iso_code_639_1.name.lower()
        except Exception as e:
            print('EXCEPTION:', text, e)
            return None

    return df[text_column].astype(str).apply(detect_lang)

fasttext

In [ ]:
def detect_fasttext_language(text):
    try:
        # Predict returns (label, probability)
        label, prob = model.predict(text.replace('\n', ' '))  # clean line breaks
        return label[0].replace("__label__", ""), prob[0]
    except Exception as e:
        print("EXCEPTION:", e)
        return None, None

create voting results

In [ ]:
# Ensure the result columns exist beforehand
def compare_languages(row):
    # Count the values
    counts = row.value_counts().to_dict()
    
    # Determine max frequency and tied languages
    max_val = max(counts.values())
    max_keys = [k for k, v in counts.items() if v == max_val]
    
    # Decide verdict
    if max_val == 2 and len(max_keys) == 1:
        verdict = max_keys[0]
    else:
        verdict = 'check_manually'
    
    return pd.Series([counts, verdict])


check for errors (to be used with test sets only, otherwise commment out)

In [ ]:
#df['comp_error'] = np.where((df['lang']!=df['verdict'])&(df['verdict']!='check_manually'),1,0)

save output table

In [ ]:
def processcommentfile(df):
    df['text_nourls'] = df['textOriginal'].str.replace('http[s]?://[\\S]+(?=\s|$)',' ')
    print('detecting lingua')
    df['lang_lingua'] = detect_language_lingua(df, 'text_nourls', detector_lingua)
    print('detecting fast')
    df[['lang_fast', 'lang_fast_confidence']] = df['text_nourls'].astype(str).apply(
        lambda x: pd.Series(detect_fasttext_language(x)))
    print('comparing')
    df['comp_results'] = None
    df['comp_verdict'] = None

    df[['comp_results', 'comp_verdict']] = df[['lang_lingua', 'lang_fast']].apply(compare_languages, axis=1)
    df = df.drop(columns=['text_nourls'])
    return df

# Execute language processing

In [ ]:
for game in directories.keys():
    print(f"Processing game: {game}")
    for dir in directories[game]['wd']:
        print('inspecting dir', dir)
        for file in identify_datafiles_in_directory(dir, 'comments_cleaned.csv'):
            print('processing file', file)
            saveto = f"{file.split('.csv')[0]}_langdetect.csv"
            print(f'saving to: {saveto}')
            data = pd.read_csv(os.path.join(dir, file), engine='python', sep=',')
            results = processcommentfile(data)
            results.to_csv(os.path.join(dir, saveto), index=False)   


# calculating and adding language diversity results to metadata files

In [ ]:
def count_verdicts(group):
    return dict(Counter(group))

def homogeneity_score(verdict_dict):
    # Remove 'check_manually' if present
    filtered = {lang: count for lang, count in verdict_dict.items() if lang != 'check_manually'}
    total = sum(filtered.values())
    if total == 0:
        return 0.0  # or np.nan if you prefer
    max_count = max(filtered.values())
    return max_count / total

for game in directories.keys():
    for dir in directories[game]['wd']:
        for file in identify_datafiles_in_directory(dir, 'comments_cleaned_langdetect.csv'):
            filelabel = file.split('_comments_cleaned_langdetect.csv')[0]
            dftmp = pd.read_csv(os.path.join(dir, file), usecols=['videoId', 'comp_verdict'])
            saveto = f"{file.split('_comments_cleaned_langdetect.csv')[0]}_languages_perVideoId2.csv"
            counts_df = dftmp.groupby('videoId')['comp_verdict'].agg(count_verdicts).reset_index()
            counts_df['num_languages'] = counts_df['comp_verdict'].apply(len)
            counts_df['language_homogeneity'] = counts_df['comp_verdict'].apply(homogeneity_score)
            print(counts_df.head())
            counts_df.to_csv(os.path.join(dir, saveto), index=False)
            metafile = f'{filelabel}_metadata_cleaned.csv'
            print('attempting to merge data with metadata', metafile)
            try:
                tmpmeta = pd.read_csv(os.path.join(dir, metafile))
                saveto = f"{filelabel}_metadata_cleaned_langinfo.csv"
                result = pd.merge(tmpmeta, counts_df, how='left', on=['videoId'])
                result.to_csv(os.path.join(dir, saveto), index=False)
            except:
                print('data merging failed')
